In [1]:
import json
import pandas as pd
from IPython.display import Markdown, display
from pathlib import Path

input_dir = Path("../data")
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

# Data Cleaning & Exploration

From the original dataset, we have removed:
- two rows, which are developer maps,
- 1,400 rows, because they are not full 8v8 matches.

After running this notebook, the cleaned matches are saved in /data/cleaned/matches.parquet. Use that file by reading it with pandas like in the example below:
```python
import pandas as pd

# Load the cleaned dataset directly
df = pd.read_parquet("../data/cleaned/matches.parquet")

# Inspect the ready-to-use data
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")
df.info()
```

In [2]:
df = pd.read_csv(input_dir / "all_matches_07_11_26.csv")

def parse_players(val):
  if pd.isna(val):
    return []
  try:
    if isinstance(val, str):
      val = val.replace('""', '"')
    return json.loads(val)
  except Exception:
    return []


df["players_parsed"] = df["players"].apply(parse_players)

In [3]:
def evaluate_match(players):
  team_0_count = 0
  team_1_count = 0
  has_composition_data = True

  if not isinstance(players, list):
    return pd.Series(
        {
            "is_8v8": False,
            "has_full_tank_composition_data": False,
        }
    )

  for p in players:
    if not isinstance(p, dict):
      continue

    t_idx = p.get("teamIndex")
    if t_idx == 0:
      team_0_count += 1
    elif t_idx == 1:
      team_1_count += 1

    build = p.get("build") or {}
    build_status = build.get("status")
    components = build.get("components") or {}
    comp_status = components.get("status")

    # If any player has 'unknown' status in either build or components, 
    # we don't have full composition data for this match.
    if build_status == "unknown" or comp_status == "unknown":
      has_composition_data = False

  # Check if the match has all 16 players on the correct teams 
  is_8v8 = (team_0_count == 8) and (team_1_count == 8)

  return pd.Series(
      {
          "is_8v8": is_8v8,
          "has_full_tank_composition_data": has_composition_data,
      }
  )

match_flags = df["players_parsed"].apply(evaluate_match)
df = pd.concat([df, match_flags], axis=1)

In [ ]:
def analyze_match_dataset(df):
  total_matches = len(df)

  # 1. Match & Composition Breakdown
  matches_8v8 = df["is_8v8"].sum() if "is_8v8" in df.columns else total_matches
  matches_not_8v8 = total_matches - matches_8v8

  subset_8v8 = (
      df[df["is_8v8"] == True].copy() if "is_8v8" in df.columns else df.copy()
  )

  if "has_full_tank_composition_data" in subset_8v8.columns:
    full_comp_8v8 = subset_8v8["has_full_tank_composition_data"].sum()
  else:
    full_comp_8v8 = 0

  partial_or_missing_comp_8v8 = len(subset_8v8) - full_comp_8v8

  print("=== Match Dataset Statistics ===")
  print(f"Total Matches in Dataset        : {total_matches:,}")
  print(
      f"  - 8v8 Matches                 : {matches_8v8:,}"
      f" ({matches_8v8 / total_matches:.1%})"
  )
  print(
      f"  - Non-8v8 Matches (Filtered)  : {matches_not_8v8:,}"
      f" ({matches_not_8v8 / total_matches:.1%})"
  )

  print("\n=== Composition Data Breakdown (8v8 Matches Only) ===")
  print(f"Total Valid 8v8 Matches         : {len(subset_8v8):,}")
  if len(subset_8v8) > 0:
    print(
        f"  - With Full Composition Data  : {full_comp_8v8:,}"
        f" ({full_comp_8v8 / len(subset_8v8):.1%})"
    )
    print(
        f"  - With Partial/Missing Data   : {partial_or_missing_comp_8v8:,}"
        f" ({partial_or_missing_comp_8v8 / len(subset_8v8):.1%})"
    )

  # Map Breakdown & Win Rates
  if (
      "mapDisplayName" in subset_8v8.columns
      and "winningTeamIndex" in subset_8v8.columns
  ):
    map_stats = (
        subset_8v8.groupby("mapDisplayName")
        .agg(
            total_matches=("matchGuid", "count"),
            team_0_wins=("winningTeamIndex", lambda x: (x == 0).sum()),
            team_1_wins=("winningTeamIndex", lambda x: (x == 1).sum()),
        )
        .reset_index()
    )

    map_stats["team_0_win_rate"] = (
        map_stats["team_0_wins"] / map_stats["total_matches"]
    )
    map_stats["team_1_win_rate"] = (
        map_stats["team_1_wins"] / map_stats["total_matches"]
    )
    map_stats = map_stats.sort_values(by="total_matches", ascending=False)

    map_display = map_stats.copy()
    map_display["total_matches"] = map_display["total_matches"].apply(
        lambda x: f"{x:,}"
    )
    map_display["team_0_wins"] = map_display["team_0_wins"].apply(
        lambda x: f"{x:,}"
    )
    map_display["team_1_wins"] = map_display["team_1_wins"].apply(
        lambda x: f"{x:,}"
    )
    map_display["team_0_win_rate"] = map_display["team_0_win_rate"].apply(
        lambda x: f"{x:.1%}"
    )
    map_display["team_1_win_rate"] = map_display["team_1_win_rate"].apply(
        lambda x: f"{x:.1%}"
    )

    display(Markdown("### 🗺️ Map Breakdown & Win Rates"))
    display(Markdown(map_display.to_markdown(index=False)))
  else:
    map_stats = pd.DataFrame()

  # Tank Usage, Relative Pick Rate & Win Rates
  player_rows = []
  if (
      "players_parsed" in subset_8v8.columns
      and "winningTeamIndex" in subset_8v8.columns
  ):
    for _, row in subset_8v8.iterrows():
      winning_team = row["winningTeamIndex"]
      for p in row.get("players_parsed", []):
        if not isinstance(p, dict):
          continue
        team_idx = p.get("teamIndex")
        player_rows.append(
            {
                "matchGuid": row.get("matchGuid"),
                "vehicleName": p.get("vehicleName", "Unknown"),
                "won": 1 if team_idx == winning_team else 0,
                "damage": p.get("damage", 0),
                "destroyed": p.get("destroyed", 0),
            }
        )

  players_flat_df = pd.DataFrame(player_rows)
  if not players_flat_df.empty:
    total_tank_selections = len(players_flat_df)

    tank_stats = (
        players_flat_df.groupby("vehicleName")
        .agg(
            total_picks=("matchGuid", "count"),
            total_wins=("won", "sum"),
            avg_damage=("damage", "mean"),
            avg_destroyed=("destroyed", "mean"),
        )
        .reset_index()
    )

    tank_stats["relative_pick_rate"] = (
        tank_stats["total_picks"] / total_tank_selections
    )
    tank_stats["win_rate"] = (
        tank_stats["total_wins"] / tank_stats["total_picks"]
    )
    tank_stats = tank_stats.sort_values(by="total_picks", ascending=False)

    # Format numbers for display copy
    tank_display = tank_stats.copy()
    tank_display["total_picks"] = tank_display["total_picks"].apply(
        lambda x: f"{x:,}"
    )
    tank_display["relative_pick_rate"] = tank_display[
        "relative_pick_rate"
    ].apply(lambda x: f"{x:.1%}")
    tank_display["win_rate"] = tank_display["win_rate"].apply(
        lambda x: f"{x:.1%}"
    )
    tank_display["avg_damage"] = tank_display["avg_damage"].apply(
        lambda x: f"{x:.1f}"
    )
    tank_display["avg_destroyed"] = tank_display["avg_destroyed"].apply(
        lambda x: f"{x:.2f}"
    )

    display(Markdown("### 🛡️ Tank Stats, Pick Rates & Win Rates"))
    display(Markdown(tank_display.to_markdown(index=False)))
  else:
    tank_stats = pd.DataFrame()

  return {
      "map_stats": map_stats,
      "tank_stats": tank_stats,
  }

res = analyze_match_dataset(df)

=== Match Dataset Statistics ===
Total Matches in Dataset        : 50,071
  - 8v8 Matches                 : 48,671 (97.2%)
  - Non-8v8 Matches (Filtered)  : 1,400 (2.8%)

=== Composition Data Breakdown (8v8 Matches Only) ===
Total Valid 8v8 Matches         : 48,671
  - With Full Composition Data  : 37,850 (77.8%)
  - With Partial/Missing Data   : 10,821 (22.2%)


### 🗺️ Map Breakdown & Win Rates

| mapDisplayName   |   total_matches |   team_0_wins |   team_1_wins | team_0_win_rate   | team_1_win_rate   |
|:-----------------|----------------:|--------------:|--------------:|:------------------|:------------------|
| Fields           |           8,907 |         4,604 |         4,303 | 51.7%             | 48.3%             |
| Divide           |           8,833 |         4,408 |         4,425 | 49.9%             | 50.1%             |
| Ravine           |           8,767 |         4,492 |         4,275 | 51.2%             | 48.8%             |
| Scorch           |           8,565 |         4,450 |         4,115 | 52.0%             | 48.0%             |
| WindValley       |           8,388 |         4,776 |         3,612 | 56.9%             | 43.1%             |
| Core             |           5,209 |         2,639 |         2,570 | 50.7%             | 49.3%             |
| Expanse          |               1 |             1 |             0 | 100.0%            | 0.0%              |
| Dunes            |               1 |             1 |             0 | 100.0%            | 0.0%              |

### 🛡️ Tank Stats, Pick Rates & Win Rates

| vehicleName   |   total_picks |   total_wins |   avg_damage |   avg_destroyed | relative_pick_rate   | win_rate   |
|:--------------|--------------:|-------------:|-------------:|----------------:|:---------------------|:-----------|
| Ark           |        87,945 |        43342 |       1605.2 |            0.71 | 11.3%                | 49.3%      |
| Arbalest      |        79,798 |        39574 |       1680.9 |            0.72 | 10.2%                | 49.6%      |
| Maul          |        65,019 |        33271 |       1318.2 |            0.67 | 8.3%                 | 51.2%      |
| Valor         |        61,147 |        29755 |        798.1 |            0.43 | 7.9%                 | 48.7%      |
| Atlas         |        53,570 |        28140 |       1637.1 |            0.68 | 6.9%                 | 52.5%      |
| Ikarus        |        53,354 |        25516 |        987   |            0.64 | 6.9%                 | 47.8%      |
| Phantom       |        51,334 |        25433 |       1216.6 |            0.64 | 6.6%                 | 49.5%      |
| Kestrel       |        50,353 |        23996 |       1017.1 |            0.55 | 6.5%                 | 47.7%      |
| Alecto        |        47,154 |        23456 |       1148.4 |            0.72 | 6.1%                 | 49.7%      |
| Tempest       |        43,375 |        21174 |       1255.2 |            0.68 | 5.6%                 | 48.8%      |
| Ranger        |        36,536 |        18952 |       1346.5 |            0.74 | 4.7%                 | 51.9%      |
| Fortis        |        36,166 |        19094 |       1635.1 |            0.73 | 4.6%                 | 52.8%      |
| Rook          |        30,656 |        15416 |       1454.1 |            0.63 | 3.9%                 | 50.3%      |
| Helio         |        26,619 |        13583 |       1345.9 |            0.6  | 3.4%                 | 51.0%      |
| Tricera       |        15,072 |         8040 |       1772.2 |            0.77 | 1.9%                 | 53.3%      |
| Vanguard      |        14,730 |         7184 |       1270.7 |            0.71 | 1.9%                 | 48.8%      |
| Seeker        |        12,983 |         6406 |       1128.2 |            0.64 | 1.7%                 | 49.3%      |
| Deadeye       |        12,925 |         7036 |       1954.4 |            0.96 | 1.7%                 | 54.4%      |

## Preprocess

In [5]:
print(df.info())
downcast = df.copy()

int_cols = downcast.select_dtypes(include=["int64"]).columns
downcast[int_cols] = downcast[int_cols].apply(pd.to_numeric, downcast="integer")

downcast = downcast.drop(columns=["players"]).copy()

print(downcast.info())

<class 'pandas.DataFrame'>
RangeIndex: 50071 entries, 0 to 50070
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   capturedAtUtc                   50071 non-null  str   
 1   gameMode                        50071 non-null  str   
 2   mapDisplayName                  50071 non-null  str   
 3   mapId                           50071 non-null  int64 
 4   matchEndConditionLabel          50071 non-null  str   
 5   matchGuid                       50071 non-null  str   
 6   matchLengthSeconds              50071 non-null  int64 
 7   players                         50071 non-null  str   
 8   winningTeamIndex                50071 non-null  int64 
 9   players_parsed                  50071 non-null  object
 10  is_8v8                          50071 non-null  bool  
 11  has_full_tank_composition_data  50071 non-null  bool  
dtypes: bool(2), int64(3), object(1), str(6)
memory usage: 3.9

## Filter Rows

In [6]:
print(f"Dataset rows before filtering: {len(df):,}")
cleaned_df = downcast.copy()

# Check for duplicates & drop them if found
cleaned_df = cleaned_df.drop_duplicates(subset=["matchGuid"]).copy()
print(f"Dataset rows remaining after removing duplicates: {len(cleaned_df):,}")

# Remove matches that are not 8v8
cleaned_df = cleaned_df[cleaned_df["is_8v8"] == True].copy()
print(f"Dataset rows remaining after removing non 8v8 matches: {len(cleaned_df):,}")

# Filter out the developer maps
cleaned_df = cleaned_df[~cleaned_df["mapDisplayName"].isin(["Expanse", "Dunes"])].copy()
print(f"Dataset rows remaining after removing Expanse and Dunes: {len(cleaned_df):,}")

# Filter out matches shorter than 30 seconds
cleaned_df = cleaned_df[cleaned_df["matchLengthSeconds"] >= 30].copy()
print(f"Dataset rows remaining after removing unrealistically short matches: {len(cleaned_df):,}")

# Keep only rows where winningTeamIndex is strictly 0 or 1
cleaned_df = cleaned_df[cleaned_df["winningTeamIndex"].isin([0, 1])].copy()
print(f"Dataset rows remaining after removing matches where the winningTeamIndex is not 0 or 1: {len(cleaned_df):,}")

Dataset rows before filtering: 50,071
Dataset rows remaining after removing duplicates: 50,071
Dataset rows remaining after removing non 8v8 matches: 48,671
Dataset rows remaining after removing Expanse and Dunes: 48,669
Dataset rows remaining after removing unrealistically short matches: 48,669
Dataset rows remaining after removing matches where the winningTeamIndex is not 0 or 1: 48,669


In [7]:
cleaned_df.to_parquet(output_dir / "matches.parquet", index=False)
print(f"Directory ready and file saved successfully to: {output_dir / 'matches.parquet'}")

Directory ready and file saved successfully to: ../data/cleaned/matches.parquet
